# Week 2 Exercise — Technical Q&A Assistant

**By:** Jaymineh  
**API:** Z.AI (GLM Models)

This builds on the Week 1 technical question answerer and upgrades it with everything covered in Week 2:

- **Gradio UI** — a clean chat interface in the browser
- **Streaming** — responses appear token by token
- **Expert system prompt** — tuned for technical and coding questions
- **Model switching** — swap between GLM models at runtime
- **Tool use (bonus)** — the assistant can look up Python documentation on demand using `pydoc`

In [ ]:
# Imports

import os
import json
import pydoc
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
# Load environment variables and initialise the Z.AI client

load_dotenv(override=True)
api_key = os.getenv('ZAI_API_KEY')

if not api_key:
    print("ZAI_API_KEY not found — please add it to your .env file")
else:
    print(f"ZAI_API_KEY found and begins: {api_key[:8]}...")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.z.ai/api/paas/v4/"
)

In [ ]:
# Available GLM models and default selection

MODELS = [
    "glm-4.7",      # fast, great for most questions
    "glm-4-plus",   # stronger reasoning
    "glm-5",        # most capable
]

DEFAULT_MODEL = "glm-4.7"

In [ ]:
# System prompt — the assistant's personality and expertise

SYSTEM_PROMPT = """\
You are an expert technical assistant specialising in Python, software engineering, \
data science, machine learning, and LLM engineering.

Your style:
- Give precise, accurate answers with working code examples where helpful.
- Format code in fenced code blocks with the language tag (e.g. ```python).
- When explaining a concept, lead with a one-sentence summary, then go deeper.
- If a question is ambiguous, make your assumptions explicit.
- You have access to a tool called `get_python_docs` — use it whenever the user \
asks about a specific Python module, class, or function and you want to pull the \
official documentation to give the most accurate answer.
"""

In [ ]:
# Tool implementation — fetch Python stdlib docs using pydoc

def get_python_docs(topic: str) -> str:
    """
    Return plain-text Python documentation for a module, class, function,
    or method (e.g. 'os.path', 'str.join', 'collections.Counter').
    """
    try:
        result = pydoc.render_doc(topic, renderer=pydoc.plaintext)
        # Cap at 3000 characters so we don't flood the context window
        return result[:3000] if len(result) > 3000 else result
    except Exception as exc:
        return f"Could not retrieve documentation for '{topic}': {exc}"


# Quick sanity check
print(get_python_docs("os.path.join")[:300])

In [ ]:
# Tool schema — describes the tool to the model in OpenAI function-calling format

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_python_docs",
            "description": (
                "Fetch the official Python documentation for a module, class, "
                "function, or method. Use this when the user asks about a specific "
                "Python API and you want to give an accurate, up-to-date answer. "
                "Examples: 'os.path', 'collections.defaultdict', 'str.format'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The Python symbol to look up, e.g. 'itertools.chain'."
                    }
                },
                "required": ["topic"],
                "additionalProperties": False
            }
        }
    }
]

In [ ]:
# Tool dispatcher — called when the model decides to use a tool

def handle_tool_calls(tool_message) -> list[dict]:
    """
    Execute each tool call requested by the model and return the results
    as a list of tool-role messages ready to be appended to the conversation.
    """
    responses = []
    for tool_call in tool_message.tool_calls:
        fn_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if fn_name == "get_python_docs":
            result = get_python_docs(args["topic"])
            print(f"[TOOL] get_python_docs('{args['topic']}') called", flush=True)
        else:
            result = f"Unknown tool: {fn_name}"

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [ ]:
# Core chat function — handles tool calls, then streams the final reply

def chat(message: str, history: list[dict], model: str):
    """
    Gradio streaming chat callback.

    Strategy:
    1. Make a non-streaming call with tools enabled to detect any tool calls.
    2. Execute tool calls in a loop until the model is satisfied.
    3. Stream the final text response back to Gradio token by token.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += [{"role": h["role"], "content": h["content"]} for h in history]
    messages.append({"role": "user", "content": message})

    # --- Phase 1: tool-call loop (non-streaming) ---
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_responses = handle_tool_calls(tool_message)
        messages.append(tool_message)          # assistant's tool-call request
        messages.extend(tool_responses)        # tool results
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools
        )

    # --- Phase 2: stream the final text reply ---
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        if delta.content:
            result += delta.content
            yield result

In [ ]:
# Gradio UI — ChatInterface with model dropdown

model_dropdown = gr.Dropdown(
    choices=MODELS,
    value=DEFAULT_MODEL,
    label="GLM Model",
    info="Switch between Z.AI GLM models at any time."
)

demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[model_dropdown],
    title="Technical Q&A Assistant",
    description=(
        "Ask anything about Python, software engineering, data science, or LLM engineering. "
        "The assistant can look up Python documentation on the fly using its built-in tool. "
        "Select a GLM model from the dropdown on the right."
    ),
    examples=[
        ["What does `yield from` do in Python and when should I use it?"],
        ["Show me how to use collections.Counter with a practical example."],
        ["Explain the difference between __str__ and __repr__."],
        ["What is the purpose of *args and **kwargs?"],
        ["How does the GIL affect multi-threading in Python?"],
    ],
    cache_examples=False,
)

demo.launch()